# Notebook 03 — Data Cleaning

**Objective:** Apply targeted data transformations to remediate all issues identified in the validation report (Notebook 02). Produce clean, schema-conformant Parquet files ready for exploratory analysis and downstream feature engineering.

**Key question:** *Is the data now ready for analysis and modelling?*

| In Scope | Out of Scope |
|----------|-------------|
| Null handling (flagging) | Feature engineering |
| Dtype enforcement and casting | Exploratory visualizations |
| Text field normalization | Model training |
| Invalid/orphan categorical codes | Sentiment/metadata extraction |
| Persisting cleaned data as Parquet | Image preprocessing |
| Post-cleaning re-validation | — |

---
## 1 · Imports & Configuration

In [1]:
from __future__ import annotations

# ── pyarrow/pandas version compatibility fix ─────────────────────
# pandas patch_pyarrow() tries to unregister "arrow.py_extension_type"
# before it is registered, causing ArrowKeyError in some version combos.
# Setting _hotfix_installed skips that broken patch safely.
import pyarrow as _pa
if not hasattr(_pa, "_hotfix_installed"):
    try:
        _pa.unregister_extension_type("arrow.py_extension_type")
    except Exception:
        _pa._hotfix_installed = True  # skip pandas' patch; type isn't registered
# ─────────────────────────────────────────────────────────────────

import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

from adoption_accelerator import config as cfg
from adoption_accelerator.data.ingestion import (
    load_tabular,
    load_reference_table,
    save_parquet,
    load_parquet,
)
from adoption_accelerator.data.schemas import get_tabular_schema
from adoption_accelerator.data.cleaning import (
    handle_missing_names,
    normalize_text_fields,
    fix_breed_swap,
    fix_invalid_codes,
    enforce_dtypes,
)
from adoption_accelerator.data.validation import (
    validate_schema,
    validate_domain,
    validate_referential_integrity,
    check_nulls,
    check_duplicates,
    check_cross_column_consistency,
    generate_validation_report,
)
from adoption_accelerator.utils.logging import setup_logging

setup_logging()
logger = logging.getLogger("adoption_accelerator")

np.random.seed(cfg.SEED)

print(f"Seed : {cfg.SEED}")
print(f"Raw  : {cfg.DATA_RAW}")
print(f"Clean: {cfg.DATA_CLEANED}")


Seed : 42
Raw  : /workspaces/adoption_accelerator/data/raw
Clean: /workspaces/adoption_accelerator/data/cleaned


---
## 2 · Load Raw Data

In [2]:
# Load raw CSVs — these are the immutable inputs
train_raw = load_tabular("train")
test_raw  = load_tabular("test")

# Reference tables
ref_breeds = load_reference_table("breed")
ref_colors = load_reference_table("color")
ref_states = load_reference_table("state")

# Snapshot raw shapes for assertions later
RAW_TRAIN_ROWS = len(train_raw)
RAW_TEST_ROWS  = len(test_raw)

print(f"Train: {train_raw.shape}")
print(f"Test : {test_raw.shape}")
print(f"Breeds: {len(ref_breeds)} | Colors: {len(ref_colors)} | States: {len(ref_states)}")

21:07:50  INFO      Loading train data from /workspaces/adoption_accelerator/data/raw/train/train.csv
21:07:50  INFO      Loaded train: 14993 rows × 24 columns
21:07:50  INFO      Loading test data from /workspaces/adoption_accelerator/data/raw/test/test.csv
21:07:50  INFO      Loaded test: 3972 rows × 23 columns
21:07:50  INFO      Loading reference table 'breed' from /workspaces/adoption_accelerator/data/raw/breed_labels.csv
21:07:50  INFO      Loaded 'breed': 307 rows × 3 columns
21:07:50  INFO      Loading reference table 'color' from /workspaces/adoption_accelerator/data/raw/color_labels.csv
21:07:50  INFO      Loaded 'color': 7 rows × 2 columns
21:07:50  INFO      Loading reference table 'state' from /workspaces/adoption_accelerator/data/raw/state_labels.csv
21:07:50  INFO      Loaded 'state': 15 rows × 2 columns


Train: (14993, 24)
Test : (3972, 23)
Breeds: 307 | Colors: 7 | States: 15


In [3]:
# Working copies — never mutate *_raw
train_df = train_raw.copy()
test_df  = test_raw.copy()

# Collect cleaning log entries
cleaning_log: list[dict] = []

---
## 3 · Load Validation Report

The validation report from Notebook 02 drives the remediation steps below.  
We load it to confirm which issues require action.

In [4]:
val_report_path = cfg.REPORTS_DIR / "validation_report.json"
with open(val_report_path, "r", encoding="utf-8") as f:
    val_report = json.load(f)

print(f"Validation report generated at: {val_report['generated_at']}")
print(f"Overall passed: {val_report['overall_passed']}")
print(f"Checks: {val_report['passed_checks']}/{val_report['total_checks']} passed")
print()

# Highlight failed / warning checks
for r in val_report["results"]:
    if not r.get("passed", True):
        print(f"  ❌  {r['check']:40s}  (split/scope: {r.get('split', r.get('fk_column', ''))})")

Validation report generated at: 2026-02-22T01:01:07.865266+00:00
Overall passed: False
Checks: 24/27 passed

  ❌  referential_integrity                     (split/scope: Breed1)
  ❌  cross_column_consistency                  (split/scope: )


---
## 4 · Handle Missing & Noisy Names

**Strategy:** Standardize empty strings, whitespace-only values, **placeholder patterns** (e.g. "No Name", "Unnamed", "None", "N/A"), **numeric-only** strings, and **symbol-only** strings in `Name` to `NaN`.  
Create a binary `has_name` flag (1 = valid name present, 0 = absent) for downstream feature engineering.

In [5]:
# Before
print("Before — Name null counts:")
print(f"  Train: {train_df['Name'].isna().sum()}")
print(f"  Test : {test_df['Name'].isna().sum()}")

train_df, train_name_log = handle_missing_names(train_df)
cleaning_log.append({"split": "train", **train_name_log})

test_df, test_name_log = handle_missing_names(test_df)
cleaning_log.append({"split": "test", **test_name_log})

# After
print("\nAfter — Name null counts (incl. noisy → NaN):")
print(f"  Train: {train_df['Name'].isna().sum()}")
print(f"  Test : {test_df['Name'].isna().sum()}")
print(f"\nhas_name distribution (train):")
print(train_df["has_name"].value_counts().to_string())
print(f"\nNoisy conversions (train): placeholders={train_name_log.get('placeholders_converted', 0)}, "
      f"numeric={train_name_log.get('numeric_only_converted', 0)}, symbol={train_name_log.get('symbol_only_converted', 0)}")

21:08:39  INFO      handle_missing_names: 0 empty→NaN, 121 noisy→NaN (placeholder=84, numeric=8, symbol=33), 1386 total nulls, has_name flag added
21:08:39  INFO      handle_missing_names: 0 empty→NaN, 27 noisy→NaN (placeholder=26, numeric=0, symbol=1), 441 total nulls, has_name flag added


Before — Name null counts:
  Train: 1265
  Test : 414

After — Name null counts (incl. noisy → NaN):
  Train: 1386
  Test : 441

has_name distribution (train):
has_name
1    13607
0     1386

Noisy conversions (train): placeholders=84, numeric=8, symbol=33


In [6]:
train_df["Name"].isna().sum()

np.int64(1386)

---
## 5 · Normalize Text Fields

Strip leading/trailing whitespace and normalize encoding (NFC) on `Name` and `Description`.

In [7]:
text_cols = ["Name", "Description"]

train_df, log = normalize_text_fields(train_df, text_cols)
cleaning_log.append({"split": "train", **log})
print("Train text normalization:", log["changes_per_column"])

test_df, log = normalize_text_fields(test_df, text_cols)
cleaning_log.append({"split": "test", **log})
print("Test  text normalization:", log["changes_per_column"])

21:16:31  INFO      normalize_text_fields: {'Name': 0, 'Description': 144}
21:16:31  INFO      normalize_text_fields: {'Name': 0, 'Description': 7}


Train text normalization: {'Name': 0, 'Description': 144}
Test  text normalization: {'Name': 0, 'Description': 7}


---
## 6 · Handle Invalid Breed Codes

Validation found **5 rows** with `Breed1=0` (not in breed_labels).  
Among those, all 5 have `Breed2≠0` — we first **swap** `Breed2 → Breed1`, then fix any remaining orphans with a fallback (ID 307 = "Mixed Breed").

In [8]:
valid_breed_ids = set(ref_breeds["BreedID"].unique()) | {0}

# Before
print("Before — Breed1=0 counts:")
print(f"  Train: {(train_df['Breed1'] == 0).sum()}")
print(f"  Test : {(test_df['Breed1'] == 0).sum()}")

# Step 1: Swap Breed2 → Breed1 where Breed1=0 and Breed2≠0
for split_name, df in [("train", train_df), ("test", test_df)]:
    df_new, log = fix_breed_swap(df)
    cleaning_log.append({"split": split_name, **log})
    if split_name == "train":
        train_df = df_new
    else:
        test_df = df_new
    print(f"  {split_name} — swapped: {log['swapped_rows']}")

# Step 2: Fix any remaining invalid Breed1 values
for split_name, df in [("train", train_df), ("test", test_df)]:
    df_new, log = fix_invalid_codes(df, "Breed1", valid_breed_ids, fallback=307)
    cleaning_log.append({"split": split_name, **log})
    if split_name == "train":
        train_df = df_new
    else:
        test_df = df_new
    print(f"  {split_name} — Breed1 remaining fixes: {log['fixes']}")

# Step 3: Fix invalid Breed2 values
for split_name, df in [("train", train_df), ("test", test_df)]:
    df_new, log = fix_invalid_codes(df, "Breed2", valid_breed_ids, fallback=0)
    cleaning_log.append({"split": split_name, **log})
    if split_name == "train":
        train_df = df_new
    else:
        test_df = df_new
    print(f"  {split_name} — Breed2 fixes: {log['fixes']}")

print("\nAfter — Breed1=0 counts:")
print(f"  Train: {(train_df['Breed1'] == 0).sum()}")
print(f"  Test : {(test_df['Breed1'] == 0).sum()}")

21:17:08  INFO      fix_breed_swap: 5 rows swapped
21:17:08  INFO      fix_breed_swap: 0 rows swapped
21:17:08  INFO      fix_invalid_codes [Breed1]: 0 fixes (fallback=307)
21:17:08  INFO      fix_invalid_codes [Breed1]: 0 fixes (fallback=307)
21:17:08  INFO      fix_invalid_codes [Breed2]: 0 fixes (fallback=0)
21:17:08  INFO      fix_invalid_codes [Breed2]: 0 fixes (fallback=0)


Before — Breed1=0 counts:
  Train: 5
  Test : 0
  train — swapped: 5
  test — swapped: 0
  train — Breed1 remaining fixes: 0
  test — Breed1 remaining fixes: 0
  train — Breed2 fixes: 0
  test — Breed2 fixes: 0

After — Breed1=0 counts:
  Train: 0
  Test : 0


---
## 7 · Handle Invalid Color Codes

Validation found **no** invalid color codes.  
We run the fix defensively — expecting zero changes.

In [9]:
valid_color_ids = set(ref_colors["ColorID"].unique()) | {0}

for col in ["Color1", "Color2", "Color3"]:
    for split_name, df in [("train", train_df), ("test", test_df)]:
        df_new, log = fix_invalid_codes(df, col, valid_color_ids, fallback=0)
        cleaning_log.append({"split": split_name, **log})
        if split_name == "train":
            train_df = df_new
        else:
            test_df = df_new
        if log["fixes"] > 0:
            print(f"  ⚠ {split_name} {col}: {log['fixes']} fixes")

print("Color code validation — all clear (0 fixes expected).")

21:01:24  INFO      fix_invalid_codes [Color1]: 0 fixes (fallback=0)
21:01:24  INFO      fix_invalid_codes [Color1]: 0 fixes (fallback=0)
21:01:24  INFO      fix_invalid_codes [Color2]: 0 fixes (fallback=0)
21:01:24  INFO      fix_invalid_codes [Color2]: 0 fixes (fallback=0)
21:01:24  INFO      fix_invalid_codes [Color3]: 0 fixes (fallback=0)
21:01:24  INFO      fix_invalid_codes [Color3]: 0 fixes (fallback=0)


Color code validation — all clear (0 fixes expected).


---
## 8 · Handle Invalid State Codes

Validation found **no** invalid state codes.  
Defensive fix — expecting zero changes.

In [10]:
valid_state_ids = set(ref_states["StateID"].unique())
state_fallback = int(ref_states["StateID"].mode().iloc[0])

for split_name, df in [("train", train_df), ("test", test_df)]:
    df_new, log = fix_invalid_codes(df, "State", valid_state_ids, fallback=state_fallback)
    cleaning_log.append({"split": split_name, **log})
    if split_name == "train":
        train_df = df_new
    else:
        test_df = df_new
    if log["fixes"] > 0:
        print(f"  ⚠ {split_name} State: {log['fixes']} fixes")

print("State code validation — all clear (0 fixes expected).")

21:01:29  INFO      fix_invalid_codes [State]: 0 fixes (fallback=41324)
21:01:29  INFO      fix_invalid_codes [State]: 0 fixes (fallback=41324)


State code validation — all clear (0 fixes expected).


---
## 9 · Enforce Dtypes

Cast all columns to their canonical types as defined in the schema.

In [11]:
train_schema = get_tabular_schema("train")
test_schema  = get_tabular_schema("test")

train_df, log = enforce_dtypes(train_df, train_schema)
cleaning_log.append({"split": "train", **log})
print(f"Train casts: {log['total_casts']}")
for c in log["casts"]:
    print(f"  {c['column']}: {c['from']} → {c['to']}")

test_df, log = enforce_dtypes(test_df, test_schema)
cleaning_log.append({"split": "test", **log})
print(f"\nTest casts: {log['total_casts']}")
for c in log["casts"]:
    print(f"  {c['column']}: {c['from']} → {c['to']}")

21:01:34  INFO      enforce_dtypes: 4 column(s) cast
21:01:34  INFO      enforce_dtypes: 4 column(s) cast


Train casts: 4
  Name: str → object
  RescuerID: str → object
  Description: str → object
  PetID: str → object

Test casts: 4
  Name: str → object
  RescuerID: str → object
  Description: str → object
  PetID: str → object


---
## 10 · Standardize Column Names

**Decision:** Preserve the original PascalCase column names (e.g. `AdoptionSpeed`, `PhotoAmt`) to stay consistent with the PetFinder schema definition, the reference tables, and all existing `src/` modules. The only new column is `has_name` (snake_case), which is a derived flag.

In [12]:
# Verify column integrity — no accidental renames or drops
expected_train_cols = set(train_schema.column_names) | {"has_name"}
expected_test_cols  = set(test_schema.column_names)  | {"has_name"}

assert set(train_df.columns) == expected_train_cols, (
    f"Unexpected train columns: {set(train_df.columns) ^ expected_train_cols}"
)
assert set(test_df.columns) == expected_test_cols, (
    f"Unexpected test columns: {set(test_df.columns) ^ expected_test_cols}"
)

print(f"Train columns ({len(train_df.columns)}): {sorted(train_df.columns)}")
print(f"Test  columns ({len(test_df.columns)}):  {sorted(test_df.columns)}")
print("\n✅ Column names verified — PascalCase preserved, has_name added.")

Train columns (25): ['AdoptionSpeed', 'Age', 'Breed1', 'Breed2', 'Color1', 'Color2', 'Color3', 'Description', 'Dewormed', 'Fee', 'FurLength', 'Gender', 'Health', 'MaturitySize', 'Name', 'PetID', 'PhotoAmt', 'Quantity', 'RescuerID', 'State', 'Sterilized', 'Type', 'Vaccinated', 'VideoAmt', 'has_name']
Test  columns (24):  ['Age', 'Breed1', 'Breed2', 'Color1', 'Color2', 'Color3', 'Description', 'Dewormed', 'Fee', 'FurLength', 'Gender', 'Health', 'MaturitySize', 'Name', 'PetID', 'PhotoAmt', 'Quantity', 'RescuerID', 'State', 'Sterilized', 'Type', 'Vaccinated', 'VideoAmt', 'has_name']

✅ Column names verified — PascalCase preserved, has_name added.


---
## 11 · Post-Cleaning Re-Validation

Run the full validation suite from Notebook 02 against the cleaned DataFrames.  
All critical checks must now pass.

In [13]:
post_results: list[dict] = []

for split_name, df in [("train", train_df), ("test", test_df)]:
    # Schema validation
    post_results.append(validate_schema(df, split_name))

    # Domain validation
    post_results.append(validate_domain(df, split_name))

    # Null checks
    post_results.append(check_nulls(df))

    # Duplicate checks
    post_results.append(check_duplicates(df, "PetID"))

    # Referential integrity — Breed1
    post_results.append(
        validate_referential_integrity(df, ref_breeds, "Breed1", "BreedID")
    )
    # Referential integrity — Breed2 (allow_zero=True)
    post_results.append(
        validate_referential_integrity(df, ref_breeds, "Breed2", "BreedID", allow_zero=True)
    )
    # Referential integrity — Color1
    post_results.append(
        validate_referential_integrity(df, ref_colors, "Color1", "ColorID")
    )
    # Referential integrity — State
    post_results.append(
        validate_referential_integrity(df, ref_states, "State", "StateID")
    )

    # Cross-column consistency
    post_results.append(check_cross_column_consistency(df))

# Summary
passed = sum(1 for r in post_results if r.get("passed", True))
total  = len(post_results)
print(f"Post-cleaning validation: {passed}/{total} checks passed")
for r in post_results:
    status = "✅" if r.get("passed", True) else "❌"
    label  = r.get("check", "unknown")
    split  = r.get("split", r.get("fk_column", ""))
    print(f"  {status}  {label:40s}  {split}")

21:01:43  INFO      Schema validation [train]: PASS
21:01:43  INFO      Domain validation [train]: PASS
21:01:43  INFO      Null analysis [train]: PASS
21:01:43  INFO      Duplicate check [PetID]: PASS (0 dupes)
21:01:43  INFO      Referential integrity [Breed1 → BreedID]: 100.0% coverage (0 orphans)
21:01:43  INFO      Referential integrity [Breed2 → BreedID]: 100.0% coverage (0 orphans)
21:01:43  INFO      Referential integrity [Color1 → ColorID]: 100.0% coverage (0 orphans)
21:01:43  INFO      Referential integrity [State → StateID]: 100.0% coverage (0 orphans)
21:01:43  INFO      Schema validation [test]: PASS
21:01:43  INFO      Domain validation [test]: PASS
21:01:43  INFO      Null analysis [train]: PASS
21:01:43  INFO      Duplicate check [PetID]: PASS (0 dupes)
21:01:43  INFO      Referential integrity [Breed1 → BreedID]: 100.0% coverage (0 orphans)
21:01:43  INFO      Referential integrity [Breed2 → BreedID]: 100.0% coverage (0 orphans)
21:01:43  INFO      Referential integri

Post-cleaning validation: 18/18 checks passed
  ✅  schema_validation                         train
  ✅  domain_validation                         train
  ✅  null_analysis                             train
  ✅  duplicate_detection                       
  ✅  referential_integrity                     Breed1
  ✅  referential_integrity                     Breed2
  ✅  referential_integrity                     Color1
  ✅  referential_integrity                     State
  ✅  cross_column_consistency                  
  ✅  schema_validation                         test
  ✅  domain_validation                         test
  ✅  null_analysis                             train
  ✅  duplicate_detection                       
  ✅  referential_integrity                     Breed1
  ✅  referential_integrity                     Breed2
  ✅  referential_integrity                     Color1
  ✅  referential_integrity                     State
  ✅  cross_column_consistency                  


---
## 12 · Validation Gate

Assert all 9 critical gates (G03-1 through G03-9). Failure halts the notebook.

In [14]:
gate_results: list[tuple[str, bool, str]] = []

# G03-1: Cleaned train/test pass full schema validation
schema_ok = all(
    r.get("passed", True)
    for r in post_results
    if r.get("check") == "schema_validation"
)
gate_results.append(("G03-1", schema_ok, "Schema validation passes"))

# G03-2: No critical null violations remain
null_ok = all(
    r.get("passed", True)
    for r in post_results
    if r.get("check") == "null_check"
)
gate_results.append(("G03-2", null_ok, "No critical null violations"))

# G03-3: All referential integrity checks pass
ri_ok = all(
    r.get("passed", True)
    for r in post_results
    if r.get("check") == "referential_integrity"
)
gate_results.append(("G03-3", ri_ok, "Referential integrity passes"))

# G03-4: Cleaned train row count == raw train row count
train_rows_ok = len(train_df) == RAW_TRAIN_ROWS
gate_results.append((
    "G03-4", train_rows_ok,
    f"Train rows: {len(train_df)} == {RAW_TRAIN_ROWS}",
))

# G03-5: Cleaned test row count == raw test row count
test_rows_ok = len(test_df) == RAW_TEST_ROWS
gate_results.append((
    "G03-5", test_rows_ok,
    f"Test rows: {len(test_df)} == {RAW_TEST_ROWS}",
))

# Print gate status before persisting (G03-6..8 checked after save)
print("Validation Gates (pre-persist):")
for gid, ok, desc in gate_results:
    print(f"  {'✅' if ok else '❌'}  {gid}: {desc}")

# Assert critical gates 1-5
for gid, ok, desc in gate_results:
    assert ok, f"CRITICAL GATE {gid} FAILED: {desc}"

print("\n✅ Gates G03-1 through G03-5 passed.")

Validation Gates (pre-persist):
  ✅  G03-1: Schema validation passes
  ✅  G03-2: No critical null violations
  ✅  G03-3: Referential integrity passes
  ✅  G03-4: Train rows: 14993 == 14993
  ✅  G03-5: Test rows: 3972 == 3972

✅ Gates G03-1 through G03-5 passed.


---
## 13 · Persist Cleaned Data

Save cleaned DataFrames as Parquet files to `data/cleaned/`.

In [15]:
train_parquet_path = cfg.DATA_CLEANED / "train.parquet"
test_parquet_path  = cfg.DATA_CLEANED / "test.parquet"

save_parquet(train_df, train_parquet_path)
save_parquet(test_df, test_parquet_path)

print(f"✅ Saved: {train_parquet_path}")
print(f"✅ Saved: {test_parquet_path}")

21:01:52  INFO      Saved Parquet: /workspaces/adoption_accelerator/data/cleaned/train.parquet (14993 rows, 3.67 MB)
21:01:52  INFO      Saved Parquet: /workspaces/adoption_accelerator/data/cleaned/test.parquet (3972 rows, 0.82 MB)


✅ Saved: /workspaces/adoption_accelerator/data/cleaned/train.parquet
✅ Saved: /workspaces/adoption_accelerator/data/cleaned/test.parquet


In [16]:
# G03-6: train.parquet exists and is readable
assert train_parquet_path.exists(), f"G03-6 FAILED: {train_parquet_path} not found"
train_reload = load_parquet(train_parquet_path)
print(f"✅ G03-6: train.parquet exists and readable ({train_reload.shape})")

# G03-7: test.parquet exists and is readable
assert test_parquet_path.exists(), f"G03-7 FAILED: {test_parquet_path} not found"
test_reload = load_parquet(test_parquet_path)
print(f"✅ G03-7: test.parquet exists and readable ({test_reload.shape})")

# G03-8: Parquet round-trip integrity
assert train_reload.shape == train_df.shape, (
    f"G03-8 FAILED: train shape mismatch {train_reload.shape} != {train_df.shape}"
)
assert test_reload.shape == test_df.shape, (
    f"G03-8 FAILED: test shape mismatch {test_reload.shape} != {test_df.shape}"
)

# pyarrow reads string columns back as StringDtype ("string") instead of
# object dtype — treat "object", "string", and "str" as the same family.
_STRING_FAMILY = {"object", "string", "str"}

def _dtype_family(dt: str) -> str:
    return "string" if dt in _STRING_FAMILY else dt

for col in train_df.columns:
    expected = _dtype_family(str(train_df[col].dtype))
    actual   = _dtype_family(str(train_reload[col].dtype))
    assert expected == actual, (
        f"G03-8: dtype mismatch for '{col}': "
        f"written={train_df[col].dtype}, reloaded={train_reload[col].dtype}"
    )
print("✅ G03-8: Round-trip integrity verified (shape + dtypes)")

# G03-9: AdoptionSpeed distribution unchanged vs. raw
raw_dist   = train_raw["AdoptionSpeed"].value_counts().sort_index()
clean_dist = train_df["AdoptionSpeed"].value_counts().sort_index()
assert raw_dist.equals(clean_dist), (
    f"G03-9 FAILED: AdoptionSpeed distribution changed!\n"
    f"Raw:\n{raw_dist}\nClean:\n{clean_dist}"
)
print("✅ G03-9: AdoptionSpeed distribution unchanged")
print("\n✅ All 9 gates (G03-1 through G03-9) PASSED.")


21:01:57  INFO      Loaded Parquet: /workspaces/adoption_accelerator/data/cleaned/train.parquet (14993 rows, 25 cols)
21:01:57  INFO      Loaded Parquet: /workspaces/adoption_accelerator/data/cleaned/test.parquet (3972 rows, 24 cols)


✅ G03-6: train.parquet exists and readable ((14993, 25))
✅ G03-7: test.parquet exists and readable ((3972, 24))
✅ G03-8: Round-trip integrity verified (shape + dtypes)
✅ G03-9: AdoptionSpeed distribution unchanged

✅ All 9 gates (G03-1 through G03-9) PASSED.


---
## 14 · Cleaning Summary

Log all transformations applied with before/after metrics.

In [17]:
# Save cleaning log as JSON
cleaning_log_path = cfg.REPORTS_DIR / "cleaning_log.json"
with open(cleaning_log_path, "w", encoding="utf-8") as f:
    json.dump({
        "cleaning_steps": cleaning_log,
        "total_steps": len(cleaning_log),
        "raw_train_rows": RAW_TRAIN_ROWS,
        "cleaned_train_rows": len(train_df),
        "raw_test_rows": RAW_TEST_ROWS,
        "cleaned_test_rows": len(test_df),
        "train_columns": sorted(train_df.columns.tolist()),
        "test_columns": sorted(test_df.columns.tolist()),
    }, f, indent=2, default=str)

print(f"✅ Cleaning log saved: {cleaning_log_path}")
print(f"   Total steps logged: {len(cleaning_log)}")

✅ Cleaning log saved: /workspaces/adoption_accelerator/reports/cleaning_log.json
   Total steps logged: 20


In [18]:
# ── Before / After comparison ──
print("=" * 65)
print("           CLEANING SUMMARY — BEFORE / AFTER")
print("=" * 65)

print(f"\n{'Metric':<40} {'Before':>10} {'After':>10}")
print("-" * 65)

# Row counts
print(f"{'Train rows':<40} {RAW_TRAIN_ROWS:>10} {len(train_df):>10}")
print(f"{'Test rows':<40} {RAW_TEST_ROWS:>10} {len(test_df):>10}")

# Column counts
print(f"{'Train columns':<40} {len(train_raw.columns):>10} {len(train_df.columns):>10}")
print(f"{'Test columns':<40} {len(test_raw.columns):>10} {len(test_df.columns):>10}")

# Breed1=0 counts
print(f"{'Train Breed1=0 (orphan)':<40} {(train_raw['Breed1'] == 0).sum():>10} {(train_df['Breed1'] == 0).sum():>10}")
print(f"{'Test Breed1=0 (orphan)':<40} {(test_raw['Breed1'] == 0).sum():>10} {(test_df['Breed1'] == 0).sum():>10}")

# Cross-column: Breed2≠0 & Breed1=0
raw_cross   = int(((train_raw['Breed2'] != 0) & (train_raw['Breed1'] == 0)).sum())
clean_cross = int(((train_df['Breed2'] != 0) & (train_df['Breed1'] == 0)).sum())
print(f"{'Train Breed2≠0 & Breed1=0 violations':<40} {raw_cross:>10} {clean_cross:>10}")

# Name nulls
print(f"{'Train Name nulls':<40} {train_raw['Name'].isna().sum():>10} {train_df['Name'].isna().sum():>10}")
print(f"{'Test Name nulls':<40} {test_raw['Name'].isna().sum():>10} {test_df['Name'].isna().sum():>10}")

# has_name flag
print(f"{'has_name column added':<40} {'No':>10} {'Yes':>10}")

print("\n" + "=" * 65)
print("Data cleaning complete — all gates passed.")
print("=" * 65)

           CLEANING SUMMARY — BEFORE / AFTER

Metric                                       Before      After
-----------------------------------------------------------------
Train rows                                    14993      14993
Test rows                                      3972       3972
Train columns                                    24         25
Test columns                                     23         24
Train Breed1=0 (orphan)                           5          0
Test Breed1=0 (orphan)                            0          0
Train Breed2≠0 & Breed1=0 violations              5          0
Train Name nulls                               1265       1386
Test Name nulls                                 414        441
has_name column added                            No        Yes

Data cleaning complete — all gates passed.


---
## 15 · Conclusions

### Key Outcomes

1. **Zero row loss** — Both train (14 993) and test (3 972) retain all original rows.  
2. **Breed1 orphans resolved** — The 5 records with `Breed1=0` were fixed by swapping `Breed2 → Breed1`; all remaining breed references are now valid.  
3. **Cross-column consistency restored** — The `Breed2≠0 & Breed1=0` violation is eliminated.  
4. **Text fields normalized** — `Name` and `Description` stripped of leading/trailing whitespace, encoding normalized to NFC.  
5. **`has_name` flag created** — Binary indicator ready for downstream feature engineering.  
6. **Color & State codes** — Defensive checks confirmed zero invalid codes, as expected from validation.  
7. **Dtypes enforced** — All columns cast to canonical schema types.  
8. **AdoptionSpeed distribution preserved** — Target variable is unchanged (G03-9).  

### Artifacts Produced

| Artifact | Path | Format |
|----------|------|--------|
| Cleaned train data | `data/cleaned/train.parquet` | Parquet (snappy) |
| Cleaned test data | `data/cleaned/test.parquet` | Parquet (snappy) |
| Cleaning log | `reports/cleaning_log.json` | JSON |

### Next Steps

The cleaned Parquet files are ready for:
- **Notebook 04** — EDA: Tabular (statistical exploration)
- **Notebook 05** — EDA: Text (NLP analysis of Name/Description)
- **Notebook 06** — EDA: Images (visual analysis)